# 🧪 [Colab 실습] 가상 NPU 만들기 — MAC부터 Systolic Array까지

**온디바이스 AI 프로그래밍 · 이론 강의 「NPU의 개념과 실제 연산 동작」 연계 가상 실습**

| 항목 | 내용 |
| --- | --- |
| 실습 방식 | 실제 NPU 하드웨어 **없이**, Python으로 NPU의 내부 동작을 직접 구현·시뮬레이션 |
| 환경 | Google Colab (CPU 런타임으로 충분 — GPU 불필요) |
| 필요 패키지 | numpy, matplotlib (Colab 기본 내장) |
| 진행 방법 | **위에서부터 순서대로** 셀을 하나씩 실행 (`Shift + Enter`) — 앞 셀의 함수를 뒤에서 재사용합니다 |

## 실습 로드맵 (강의 자료 장 번호와 대응)

| Part | 주제 | 강의 자료 |
| --- | --- | --- |
| 1 | MAC 연산의 본질 — 손으로 구현하고 연산량 체감하기 | 2장 |
| 2 | Conv = GEMM — im2col 직접 구현 | 3.3절 |
| 3 | ★ Systolic Array 시뮬레이터 — 사이클 단위 추적 | 3장 |
| 4 | 데이터 이동 에너지 — DRAM이 왜 적인가 | 4~5장 |
| 5 | 타일링 — 큰 행렬을 작은 배열에서 돌리기 | 4.3절 |
| 6 | INT8 양자화 MAC — 누산기와 캘리브레이션 | 6장 |
| 7 | TOPS 계산과 Roofline 모델 | 5.2절·7장 |
| 8 | Fallback 비용 시뮬레이션 | 8장 |
| 9 | 종합 미니 프로젝트 — 나만의 가상 NPU로 Conv 레이어 실행 | 전체 |

> 💡 각 Step 끝의 **✅ 확인** 항목을 스스로 점검하고, **✏️ 직접 해보기**는 코드를 수정해 실험해 보세요.


---
# Part 0. 환경 준비

### Step 0-1. 라이브러리 불러오기

아무것도 설치할 필요가 없습니다. Colab에 기본 내장된 numpy와 matplotlib만 사용합니다.
아래 셀을 실행해 버전이 출력되면 준비 완료입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.random.seed(42)
rng = np.random.default_rng(42)

# ── 그래프 한글 폰트 설정 (Colab용 — 1분 내 완료, 실패해도 실습 진행에는 지장 없음) ──
try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"],
                   capture_output=True, timeout=120)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rc("font", family="NanumGothic")
    print("한글 폰트(NanumGothic) 설정 완료")
except Exception as e:
    print("한글 폰트 설정 생략(그래프 한글이 □로 보일 수 있으나 실습에는 무관):", e)
plt.rc("axes", unicode_minus=False)

print("numpy     :", np.__version__)
print("matplotlib:", plt.matplotlib.__version__)
print("준비 완료! 이제 가상 NPU를 만들어 봅시다 🚀")

---
# Part 1. MAC 연산의 본질 — "신경망 연산은 전부 이것 하나"

강의 자료 2장에서 배웠듯, 딥러닝 추론 연산의 90% 이상은 **MAC(Multiply-Accumulate)** 하나로 수렴합니다.

```text
acc = acc + (a × b)
```

이 Part에서는 MAC을 직접 구현하고, FC와 Conv가 정말로 MAC의 반복인지 확인한 뒤,
실제 모델 규모의 연산량이 얼마나 되는지 **몸으로 체감**합니다.

### Step 1-1. MAC 유닛을 함수로 구현하기

하드웨어 MAC 유닛은 곱셈기 1개 + 덧셈기 1개 + 누적 레지스터 1개입니다.
이를 그대로 함수로 옮기면 단 한 줄입니다.

In [ ]:
def mac(acc, a, b):
    """MAC 유닛 1개의 1사이클 동작: 누적값 + (a x b)"""
    return acc + a * b

# 동작 확인: 0에서 시작해 (5x1), (7x2)를 차례로 누적 (x·w 순서)
acc = 0
acc = mac(acc, 5, 1)   # 0 + 5*1 = 5
acc = mac(acc, 7, 2)   # 5 + 7*2 = 19
print("누적 결과:", acc)
assert acc == 19
print("✅ 이 값 19는 잠시 뒤 Part 3에서 systolic array가 만들어낼 y11과 같은 값입니다!")

### Step 1-2. FC 레이어 = MAC의 반복임을 증명하기

FC 레이어의 출력 뉴런 하나는 다음 수식이었습니다 (강의 2.2절 ①).

$$y_j = \sum_{i} \left( x_i \times w_{ij} \right) + b_j \qquad\Leftrightarrow\qquad \mathbf{y} = \mathbf{x}\,W + \mathbf{b}$$

여기서 **x:(in,) 행벡터, W:(in, out), b:(out,)** — 딥러닝 프레임워크의 `y = x @ W + b` 관례 그대로입니다.

MAC 함수만으로 FC를 구현하고, numpy 행렬곱 `x @ W + b`와 결과가 같은지 검증합니다.

In [ ]:
def fc_layer_with_mac(x, W, b):
    """MAC 함수만 사용해 FC 레이어 계산 (y = x @ W + b). x:(in,) W:(in,out) b:(out,)"""
    in_dim, out_dim = W.shape
    y = np.zeros(out_dim)
    mac_count = 0
    for j in range(out_dim):          # 출력 뉴런마다
        acc = b[j]
        for i in range(in_dim):       # 입력 개수만큼 MAC: acc += x_i * w_ij
            acc = mac(acc, x[i], W[i, j])
            mac_count += 1
        y[j] = acc
    return y, mac_count

x = rng.normal(size=8)        # 입력 8
W = rng.normal(size=(8, 4))   # (입력 8, 출력 4)
b = rng.normal(size=4)

y_mac, n_macs = fc_layer_with_mac(x, W, b)
y_np = x @ W + b

print("MAC으로 계산   :", np.round(y_mac, 4))
print("numpy x@W+b   :", np.round(y_np, 4))
print("총 MAC 횟수   :", n_macs, "(= 입력 8 x 출력 4)")
assert np.allclose(y_mac, y_np)
print("✅ FC 레이어(y = x@W + b)는 정확히 in x out 번의 MAC이다!")

### Step 1-3. Conv 출력 픽셀 1개 = 몇 번의 MAC인가

강의 2.2절 ②의 수식을 코드로 확인합니다. 3×3 커널, 입력 채널 C_in인 Conv에서
출력 픽셀 하나를 만드는 데 필요한 MAC 수는 `C_in × 3 × 3` 이어야 합니다.

> ✏️ **직접 해보기:** `C_in`을 64로 바꾸면 강의 자료 11장 문제 1의 답(576회)이 나오는지 확인하세요.

In [ ]:
def conv_one_pixel(X_patch, kernel):
    """출력 픽셀 1개 계산. X_patch, kernel: (C_in, KH, KW)"""
    acc, cnt = 0.0, 0
    C, KH, KW = kernel.shape
    for c in range(C):
        for ky in range(KH):
            for kx in range(KW):
                acc = mac(acc, kernel[c, ky, kx], X_patch[c, ky, kx])
                cnt += 1
    return acc, cnt

C_in = 3   # ✏️ 64로 바꿔보세요
patch  = rng.normal(size=(C_in, 3, 3))
kernel = rng.normal(size=(C_in, 3, 3))

val, cnt = conv_one_pixel(patch, kernel)
print(f"C_in={C_in} 3x3 Conv → 출력 픽셀 1개 = MAC {cnt}회 (공식: {C_in}x3x3={C_in*9})")
assert cnt == C_in * 9 and np.isclose(val, np.sum(patch * kernel))
print("✅ Conv도 결국 MAC의 반복이다!")

### Step 1-4. 실제 모델 규모 체감 — 왜 전용 하드웨어가 필요한가

ResNet-50 한 장 추론은 약 **41억 MAC**(강의 2.2절 ③)입니다.
Python 순수 루프로 MAC을 돌리면 초당 몇 번이나 처리할 수 있는지 실측하고,
41억 회를 처리하려면 얼마나 걸릴지 비례 계산으로 추정해 봅시다.

In [ ]:
# Python 루프의 MAC 처리 속도 실측 (100만 회)
N_TEST = 1_000_000
a_arr = rng.normal(size=N_TEST)
b_arr = rng.normal(size=N_TEST)

t0 = time.perf_counter()
acc = 0.0
for i in range(N_TEST):
    acc = acc + a_arr[i] * b_arr[i]
t1 = time.perf_counter()

macs_per_sec = N_TEST / (t1 - t0)
resnet50_macs = 4.1e9

print(f"Python 루프 실측     : {macs_per_sec:,.0f} MAC/초")
print(f"ResNet-50 1장 추론    : {resnet50_macs/macs_per_sec:,.1f} 초 (약 {resnet50_macs/macs_per_sec/60:.1f} 분!)")
print()
# 비교: numpy(내부 C+SIMD), 그리고 가상 NPU
t0 = time.perf_counter(); _ = a_arr @ b_arr; t1 = time.perf_counter()
np_speed = N_TEST / (t1 - t0)
npu_speed = 1.0e12   # 2 TOPS NPU = 초당 1조 MAC (1 MAC = 2 ops)
print(f"numpy (C+SIMD)        : {np_speed:,.0f} MAC/초  → ResNet-50 {resnet50_macs/np_speed*1000:,.0f} ms")
print(f"2 TOPS NPU (이론)     : {npu_speed:,.0f} MAC/초  → ResNet-50 {resnet50_macs/npu_speed*1000:.1f} ms")
print()
print(f"✅ NPU는 Python 대비 약 {npu_speed/macs_per_sec:,.0f}배 — 전용 하드웨어가 필요한 이유가 숫자로 보입니다.")

> **✅ Part 1 확인**
> - [ ] MAC 수식 `acc += a×b`를 설명할 수 있다
> - [ ] FC(y = x@W + b)의 MAC 수 = in×out, Conv 출력 픽셀 1개의 MAC 수 = C_in×K×K 임을 확인했다
> - [ ] 41억 MAC을 소프트웨어 루프로 도는 것이 비현실적임을 체감했다

**다음 질문:** MAC을 빠르게 하려면 병렬로 깔아야 한다. 그런데 Conv처럼 복잡한 연산을
어떻게 "MAC 배열이 처리하기 좋은 모양"으로 바꿀까? → Part 2의 im2col이 답입니다.

---
# Part 2. Conv = GEMM(General Matrix Multiplication) — im2col 직접 구현하기

강의 3.3절: Conv를 **im2col** 변환으로 거대한 행렬 곱(GEMM) 하나로 바꿀 수 있습니다.
본 노트북은 FC와 마찬가지로 **`y = x @ W` 관례**로 통일합니다 — 패치 하나가 입력 행벡터 x 하나가 됩니다.

```text
입력 X: (C_in, H, W)                X_col = im2col(X) : (OH·OW) × (C_in·K·K)   ← 패치 1개 = 행 1개
커널 W: (C_out, C_in, K, K)   →     W_mat = 커널 행렬  : (C_in·K·K) × (C_out)   ← 커널 1개 = 열 1개
                                    Y = X_col @ W_mat : (OH·OW) × (C_out)      ← y = x @ W 형태!
```

### Step 2-1. 기준이 될 직접 Conv 구현 (느리지만 확실한 정답지)

In [ ]:
def conv2d_direct(X, Wt):
    """교과서식 Conv (stride=1, padding=0). X:(C,H,W)  Wt:(OC,C,KH,KW)"""
    C, H, Wd = X.shape
    OC, _, KH, KW = Wt.shape
    OH, OW = H - KH + 1, Wd - KW + 1
    Y = np.zeros((OC, OH, OW))
    for oc in range(OC):
        for oy in range(OH):
            for ox in range(OW):
                Y[oc, oy, ox] = np.sum(Wt[oc] * X[:, oy:oy+KH, ox:ox+KW])
    return Y

# 작은 예제 입력 준비 (정수로 만들어 눈으로 검증하기 쉽게)
X_img  = rng.integers(-3, 4, (2, 5, 5)).astype(np.int64)   # 채널 2, 5x5
W_conv = rng.integers(-3, 4, (3, 2, 3, 3)).astype(np.int64) # 출력채널 3, 3x3 커널

Y_ref = conv2d_direct(X_img, W_conv)
print("정답 Conv 출력 shape:", Y_ref.shape, "(OC, OH, OW)")

### Step 2-2. im2col 구현 — 패치를 행벡터로 펼치기

커널이 훑는 각 위치의 패치를 하나의 **행**으로 펼칩니다 (`y = x @ W`의 x가 행벡터이므로).
겹치는 패치 때문에 데이터가 중복 복사되는 것이 특징이며(강의에서 언급),
NPU는 이 중복을 on-chip SRAM 재사용으로 흡수합니다.

In [ ]:
def im2col(X, KH, KW):
    """X:(C,H,W) → rows:(OH*OW, C*KH*KW) — 패치 1개가 행(row) 1개 (x@W 관례)"""
    C, H, Wd = X.shape
    OH, OW = H - KH + 1, Wd - KW + 1
    rows = np.zeros((OH * OW, C * KH * KW), dtype=X.dtype)
    idx = 0
    for oy in range(OH):
        for ox in range(OW):
            rows[idx] = X[:, oy:oy+KH, ox:ox+KW].reshape(-1)
            idx += 1
    return rows, OH, OW

X_col, OH, OW = im2col(X_img, 3, 3)
print("im2col 결과 shape :", X_col.shape, "= (OH·OW, C_in·K·K) =", (OH*OW, 2*3*3))
orig  = X_img.size
after = X_col.size
print(f"데이터 중복 배수  : {after/orig:.1f}배 (원본 {orig} → 펼친 후 {after} 원소)")

### Step 2-3. Conv ≡ GEMM 검증

커널을 (C·K·K, OC) 행렬로 펴서(커널 1개 = 열 1개) im2col 결과와 `X_col @ W_mat`로 곱한 뒤,
직접 Conv와 완전히 같은지 확인합니다.

In [ ]:
W_mat  = W_conv.reshape(3, -1).T                 # (C_in·K·K, OC) — 커널 1개가 열 1개
Y_gemm = (X_col @ W_mat).T.reshape(3, OH, OW)    # (OH·OW, OC) → (OC, OH, OW)

print("GEMM으로 계산한 Conv와 직접 Conv 일치? →", np.array_equal(Y_gemm, Y_ref))
assert np.array_equal(Y_gemm, Y_ref)
print("✅ Conv = 행렬곱(GEMM) 하나! 이제 '행렬곱만 잘하는 하드웨어'를 만들면 Conv도 해결됩니다.")
print()
print("X_col(im2col) shape:", X_col.shape, " W_mat(커널 행렬) shape:", W_mat.shape)
print("→ 이 X_col, W_mat을 Part 3의 systolic array에 그대로 넣을 예정입니다.")

> **✅ Part 2 확인**
> - [ ] im2col이 패치를 행으로 펼쳐 `Y = X_col @ W_mat` 하나로 만드는 과정을 설명할 수 있다
> - [ ] Conv가 GEMM 한 번과 수학적으로 동일함을 코드로 검증했다
> - [ ] 데이터 중복(≈K² 배)이 생기는 이유를 이해했다

---
# Part 3. ★ Systolic Array 시뮬레이터 — NPU의 심장을 직접 만들기

이 Part가 본 실습의 핵심입니다. 강의 3장의 **Weight-Stationary Systolic Array**를
사이클 단위로 동작하는 시뮬레이터로 구현하고, 강의 자료의 2×2 손계산 예제와
**한 사이클도 다르지 않게 일치**하는지 확인합니다.

> 📐 **표기 주의:** 강의 자료는 `Y = W·X` 표기지만, 본 노트북은 FC·Conv와 동일한
> 프레임워크 관례 **`Y = X @ W`**로 통일했습니다. 행렬이 전치되어 배열의 행·열 역할이
> 서로 바뀔 뿐(강의 그림을 90° 뒤집은 모양), **MAC에 들어가는 값과 사이클은 완전히 동일**합니다.

```text
              psum (열 상단에서 0 주입, 아래로 흐르며 누적)
                 ↓                   ↓
              ┌────────┐  ┌────────┐
  act(입력) →│ PE(0,0)        │→│ PE(0,1)        │   ← act(활성값): 행 왼쪽에서 주입, 오른쪽으로 흐름
              │ w=W[0,0]       │  │ w=W[0,1]       │      weight는 PE에 사전 로드되어 고정
              └────────┘  └────────┘
                 ↓                   ↓
              ┌────────┐  ┌────────┐
  act(입력) →│ PE(1,0)        │  │ PE(1,1)        │
              └────────┘  └────────┘
                 ↓                   ↓
               Y[·,0]              Y[·,1]        ← 결과: 열 아래 끝에서 배출
```

### Step 3-1. 동작 규칙을 말로 먼저 정리하기 (코딩 전 설계)

`Y = X @ W` (X: M×K — 입력 행벡터 M개, W: K×N)를 **K×N 크기의 PE 배열**로 계산합니다.

| 데이터 | 이동 방향 | 주입 시점 (skew가 핵심!) |
| --- | --- | --- |
| weight `W[k,j]` | 이동 없음 — PE(k,j)에 **고정** | 연산 시작 전 1회 로드 |
| activation `X[i,k]` | 행 k를 따라 **오른쪽으로** 1칸/사이클 | 사이클 `i+k` 에 행 k 왼쪽 주입 |
| psum (출력 `Y[i,j]`용) | 열 j를 따라 **아래로** 1칸/사이클 | 사이클 `i+j` 에 열 j 상단에 0으로 주입 |
| 결과 `Y[i,j]` | 열 j 아래 끝에서 배출 | 사이클 `i+j+K-1` 에 완성 |

**타이밍이 맞는 이유:** psum(i,j)는 사이클 `i+j+k`에 PE(k,j)에 도착하고,
activation X[i,k]도 (행 k에 `i+k`에 주입되어 j칸 이동해) 정확히 `i+k+j`에 같은 PE에 도착합니다.
**두 데이터가 약속 시간에 만나도록 skew를 준 것** — 이것이 systolic array 설계의 전부입니다.

### Step 3-2. 시뮬레이터 구현

각 PE는 매 사이클 `(왼쪽에서 온 act, 위에서 온 psum)`을 받아 MAC 1회를 수행하고 전달합니다.
어떤 출력 행(i)에 속한 psum인지 태그를 붙여 추적합니다.

In [ ]:
def systolic_matmul(X, W, trace=False):
    """Y = X @ W 를 Weight-Stationary systolic array로 사이클 단위 시뮬레이션.
    X:(M,K) 입력 행벡터 M개, W:(K,N) 가중치 — PE(k,j)에 W[k,j] 고정 (K×N 배열).
    반환: Y, 총사이클, 사이클별 MAC 수, (trace=True면) 전체 이벤트 로그
    로그 형식: (cycle, k, j, weight, act, psum_in, psum_out)
    """
    M, K = X.shape
    K2, N = W.shape
    assert K == K2, "X의 열 수와 W의 행 수가 같아야 합니다"
    total_cycles = (M - 1) + (N - 1) + (K - 1) + 1   # 파이프라인 총 길이
    Y = np.zeros((M, N), dtype=np.int64)
    act  = [[None]*N for _ in range(K)]   # 각 PE가 들고 있는 (act값, 출력행 i)
    psum = [[None]*N for _ in range(K)]   # 각 PE가 들고 있는 (psum값, 출력행 i)
    log, mac_per_cycle = [], []

    for t in range(total_cycles):
        new_act  = [[None]*N for _ in range(K)]
        new_psum = [[None]*N for _ in range(K)]
        macs = 0
        for k in range(K):
            for j in range(N):
                # ① 왼쪽에서 들어오는 activation 수신 (맨 왼쪽 열은 스케줄에 따라 주입)
                if j == 0:
                    i = t - k                       # skew: X[i,k]는 사이클 i+k에 주입
                    a = (int(X[i, k]), i) if 0 <= i < M else None
                else:
                    a = act[k][j-1]                 # 왼쪽 PE가 지난 사이클에 갖고 있던 값
                new_act[k][j] = a
                if a is None:
                    continue
                # ② 위에서 내려오는 psum 수신 (맨 윗행은 0으로 신규 주입)
                p_in = (0, a[1]) if k == 0 else psum[k-1][j]
                if p_in is None or p_in[1] != a[1]:
                    continue                        # 약속된 짝이 아니면 대기
                # ③ MAC 1회 수행 (acc += x · w)
                val = p_in[0] + a[0] * int(W[k, j])
                new_psum[k][j] = (val, a[1])
                macs += 1
                if trace:
                    log.append((t, k, j, int(W[k, j]), a[0], p_in[0], val))
                # ④ 맨 아랫행이면 결과 배출
                if k == K - 1:
                    Y[a[1], j] = val
        act, psum = new_act, new_psum
        mac_per_cycle.append(macs)
    return Y, total_cycles, mac_per_cycle, log

print("시뮬레이터 정의 완료 — 다음 셀에서 강의 예제로 검증합니다.")

### Step 3-3. 강의 자료 2×2 예제 재현 — 손계산과 대조하기 ★

강의 3.2절에서 손으로 추적했던 바로 그 예제입니다. 강의는 `W·X` 표기였으므로,
`X @ W` 관례에서는 **모든 행렬을 전치해서** 넣습니다 (입력 벡터가 행이 됨).

```text
강의(W·X):  W = | 1  2 |   X = | 5  6 |   Y = | 19  22 |
                | 3  4 |       | 7  8 |       | 43  50 |

여기(X@W):  X = | 5  7 |   W = | 1  3 |   Y = | 19  43 |   ← 강의 Y의 전치
                | 6  8 |       | 2  4 |       | 22  50 |      (y11=19 위치는 그대로!)
```

강의 표에서 **"t2에 psum 5 + 2×7 = 19를 완성"** 했었습니다. 여기서도 같은 사이클에
w=2를 고정한 PE가 act 7을 받아 `5 + 7×2 = 19`를 완성하는지 트레이스로 확인하세요.
(시뮬레이터는 t0부터 세므로, 강의의 t1·t2·t3 = 여기의 t0·t1·t2 입니다.)

In [ ]:
X2 = np.array([[5, 7], [6, 8]])
W2 = np.array([[1, 3], [2, 4]])

Y2, cyc, mpc, log = systolic_matmul(X2, W2, trace=True)

print("시뮬레이터 결과 Y =")
print(Y2)
print("numpy 정답 X@W   =")
print(X2 @ W2)
assert np.array_equal(Y2, X2 @ W2)
print(f"\n총 사이클 수: {cyc}  (공식 M+N+K-2+1 = 2+2+2-2+1 = 4 ✓)")
print("\n───────── 사이클별 전체 트레이스 ─────────")
print(f"{'cycle':>5} {'PE':>7} {'w':>3} {'act':>4} {'psum_in':>8} {'psum_out':>9}  비고")
K_arr = W2.shape[0]
for t, k, j, w, a, p, v in log:
    note = ""
    if k == K_arr - 1:                     # 맨 아랫행 = 출력 완성
        i = t - j - (K_arr - 1)            # 배출 시점 공식의 역산
        note = f"→ Y[{i}][{j}] = {v} 완성!"
    print(f"{t:>5} PE({k},{j})  {w:>3} {a:>4} {p:>8} {v:>9}  {note}")
print()
print("👀 t1 행을 보세요: PE(1,0) w=2 a=7 psum_in=5 → 19  ← 강의 표의 'y11=19 완성'과 동일!")

### Step 3-4. 무작위 행렬로 일반 검증 + 사이클 공식 확인

시뮬레이터가 어떤 크기에서도 정확한지, 그리고 총 사이클이 공식
`M + N + K - 2` (마지막 +1 포함 시 -2+1)와 맞는지 확인합니다.

In [ ]:
print(f"{'M':>3} {'K':>3} {'N':>3} | {'사이클':>6} {'공식값':>6} | 일치")
print("-" * 40)
for _ in range(6):
    M, K, N = rng.integers(1, 8, 3)
    Xr = rng.integers(-5, 6, (M, K))
    Wr = rng.integers(-5, 6, (K, N))
    Yr, cyc, _, _ = systolic_matmul(Xr, Wr)
    ok = np.array_equal(Yr, Xr @ Wr)
    formula = (M-1) + (N-1) + (K-1) + 1
    print(f"{M:>3} {K:>3} {N:>3} | {cyc:>6} {formula:>6} | {'✅' if ok and cyc==formula else '❌'}")
    assert ok
print("\n✅ 모든 크기에서 numpy와 완전 일치 — 우리의 가상 NPU는 정확합니다.")

### Step 3-5. 파이프라인 충전·배출 시각화 — "왜 batch=1 실측 TOPS는 낮은가"

사이클별 동시 MAC 수를 그리면 사다리꼴이 나옵니다.
앞뒤의 경사 구간(파이프라인 충전/배출)에서는 PE가 놀고 있습니다.
강의 7.2절 "batch=1 실측은 2~5배 낮다"의 하드웨어적 이유가 바로 이것입니다.

In [ ]:
M, K, N = 24, 8, 8                      # 입력 행 24개가 8x8 PE 배열을 통과
Xb = rng.integers(-3, 4, (M, K))
Wb = rng.integers(-3, 4, (K, N))
Yb, cyc, mpc, _ = systolic_matmul(Xb, Wb)
assert np.array_equal(Yb, Xb @ Wb)

peak = K * N                             # PE 배열 크기 = KxN
util = sum(mpc) / (cyc * peak)

plt.figure(figsize=(9, 3.5))
plt.fill_between(range(cyc), mpc, step="mid", alpha=0.3)
plt.plot(range(cyc), mpc, drawstyle="steps-mid", lw=2)
plt.axhline(peak, ls="--", c="r", label=f"이론 최대 = KxN = {peak} MAC/cycle")
plt.xlabel("cycle"); plt.ylabel("동시 MAC 수")
plt.title(f"파이프라인 충전 → 정상 가동 → 배출   (평균 활용률 {util*100:.0f}%)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"총 MAC {sum(mpc)}회 / 이론상 최대 {cyc}x{peak}={cyc*peak}회 → 활용률 {util*100:.1f}%")
print("✏️ 직접 해보기: M을 240으로 키우면 활용률이 어떻게 변하나요? (입력 행이 길수록 충전 비중↓)")

### Step 3-6. Part 2와 연결 — Conv를 systolic array에서 실행하기

im2col로 만든 X_col, W_mat 행렬을 시뮬레이터에 그대로 넣어,
**"Conv → GEMM → systolic array"** 의 전체 사슬을 완성합니다.

In [ ]:
Y_sys, cyc, mpc, _ = systolic_matmul(X_col, W_mat)
Y_sys_img = Y_sys.T.reshape(3, OH, OW)     # (OH·OW, OC) → (OC, OH, OW)

print("systolic Conv == 직접 Conv ? →", np.array_equal(Y_sys_img, Y_ref))
assert np.array_equal(Y_sys_img, Y_ref)
M_ = X_col.shape[0]; K_, N_ = W_mat.shape
print(f"\nPE 배열 {K_}x{N_} (가중치 고정), GEMM 크기 ({M_}x{K_})·({K_}x{N_})")
print(f"총 {cyc} 사이클에 Conv 레이어 하나 완료 (총 MAC {sum(mpc)}회)")
print("✅ NPU가 Conv를 처리하는 실제 경로를 끝까지 재현했습니다!")

> **✅ Part 3 확인**
> - [ ] weight/act/psum 세 데이터의 이동 방향과 skew 주입 시점을 설명할 수 있다
> - [ ] 2×2 예제 트레이스가 강의 손계산과 일치함을 확인했다
> - [ ] 총 사이클 = M+N+K-2 공식과 파이프라인 활용률 개념을 이해했다
> - [ ] Conv → im2col → systolic array 전체 경로를 실행해 봤다
>
> ✏️ **도전:** `systolic_matmul`의 로그를 이용해 강의의 y21=43 — x@W 관례에서는 전치되어
> `Y[0][1]=43` — 이 완성되는 사이클을 찾아보세요 (강의 11장 문제 3).

---
# Part 4. 데이터 이동 에너지 — 진짜 적은 연산이 아니라 DRAM이다

강의 4.1절의 에너지 표를 그대로 시뮬레이션에 넣어,
**같은 행렬곱**을 (a) 아무 재사용 없이 매번 DRAM에서 읽는 naive 방식과
(b) weight-stationary 재사용 방식으로 실행했을 때 에너지가 몇 배 차이 나는지 계산합니다.

| 동작 | 상대 에너지 (INT8 덧셈 = 1 기준) |
| --- | --- |
| MAC (곱셈+덧셈) | 7 |
| SRAM 읽기 | 150 |
| **DRAM 읽기** | **6,000** |

### Step 4-1. 접근 횟수 세기

In [ ]:
ENERGY = {"mac": 7, "sram": 150, "dram": 6000}

def count_naive(M, K, N):
    """재사용 없음: MAC마다 X와 W를 DRAM에서 새로 읽음"""
    macs = M * K * N
    return {"mac": macs, "dram": 2 * macs, "sram": 0}   # x읽기 + w읽기

def count_weight_stationary(M, K, N):
    """WS systolic: W(K×N)는 DRAM→PE 1회, X(M×K)는 DRAM→SRAM 1회 후 SRAM에서 재사용"""
    macs = M * K * N
    return {
        "mac":  macs,
        "dram": K * N + M * K,        # W 전체 1회 + X 전체 1회
        "sram": M * K * N,            # X 원소가 N개 열 PE로 흘러갈 때 SRAM 읽기
    }

def energy(cnt):
    return sum(cnt[k] * ENERGY[k] for k in cnt)

M, K, N = 64, 64, 256   # 중간 크기 레이어 가정
naive = count_naive(M, K, N)
ws    = count_weight_stationary(M, K, N)

print(f"GEMM 크기: ({M}x{K})·({K}x{N})  → 총 MAC {M*K*N:,}회\n")
print(f"{'':14}{'DRAM 접근':>12}{'SRAM 접근':>12}{'에너지(상대)':>16}")
print(f"{'naive':14}{naive['dram']:>12,}{naive['sram']:>12,}{energy(naive):>16,}")
print(f"{'WS 재사용':13}{ws['dram']:>12,}{ws['sram']:>12,}{energy(ws):>16,}")
print(f"\n💡 DRAM 접근 {naive['dram']/ws['dram']:,.0f}배 감소 → 총 에너지 {energy(naive)/energy(ws):.1f}배 절감")

### Step 4-2. 에너지 구성 시각화 — 어디에 에너지가 쓰이는가

In [ ]:
labels = ["naive\n(재사용 없음)", "Weight-Stationary\n(systolic 재사용)"]
comps  = ["mac", "sram", "dram"]
colors = {"mac": "#4c72b0", "sram": "#dd8452", "dram": "#c44e52"}

fig, ax = plt.subplots(figsize=(8, 4.5))
for x, cnt in enumerate([naive, ws]):
    bottom = 0
    for c in comps:
        v = cnt[c] * ENERGY[c]
        ax.bar(x, v, bottom=bottom, color=colors[c], label=c.upper() if x == 0 else None)
        bottom += v
ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
ax.set_ylabel("총 에너지 (상대 단위)")
ax.set_title("같은 행렬곱, 데이터 이동 전략만 바꿨을 때의 에너지")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

n_e, w_e = energy(naive), energy(ws)
print(f"naive 에너지의 {naive['dram']*ENERGY['dram']/n_e*100:.0f}%가 DRAM — 연산(MAC)은 {naive['mac']*ENERGY['mac']/n_e*100:.1f}%뿐!")
print("✅ 'NPU 설계는 DRAM 접근 줄이기 게임'(강의 4.1절)이라는 말이 숫자로 증명됩니다.")

> **✅ Part 4 확인**
> - [ ] DRAM 접근 1회가 MAC 1회보다 압도적으로 비싼 이유를 수치로 설명할 수 있다
> - [ ] Weight-Stationary가 DRAM 접근을 어떻게 줄이는지 카운트로 확인했다
>
> ✏️ **직접 해보기:** Output-Stationary라면 어떤 접근이 줄고 어떤 접근이 늘까요?
> `count_output_stationary` 함수를 직접 작성해 비교해 보세요. (힌트: psum이 PE에 고정 → psum의 SRAM 왕복이 사라지고, 대신 weight가 매번 흘러 들어와야 함)

---
# Part 5. 타일링(Tiling) — 배열보다 큰 행렬 처리하기

실제 레이어는 PE 배열보다 훨씬 큽니다 (강의 4.3절).
**8×8 행렬곱을 4×4 PE 배열**로 처리하려면 행렬을 2×2 블록으로 쪼개고,
psum을 SRAM에 쌓아가며 타일 GEMM을 순차 실행해야 합니다.

```text
Y = | X11 X12 | | W11 W12 |      Y11 = X11·W11 + X12·W21   ← 타일 GEMM 2회 + psum 누적
    | X21 X22 | | W21 W22 |      ...  총 2x2x2 = 8회의 타일 GEMM
```

### Step 5-1. 타일링 실행기 구현 — 우리 systolic 시뮬레이터를 그대로 재사용

In [ ]:
def tiled_systolic_matmul(X, W, tile):
    """Y = X @ W ((M,K)·(K,N))를 tile x tile PE 배열로 타일링 실행.
    반환: Y, 타일GEMM 횟수, 총 사이클, psum SRAM 누적 횟수"""
    M, K = X.shape; _, N = W.shape
    assert M % tile == 0 and K % tile == 0 and N % tile == 0, "간단히 나누어떨어지는 경우만"
    Y = np.zeros((M, N), dtype=np.int64)
    n_tile_gemm = total_cycles = psum_accum = 0
    for i0 in range(0, M, tile):
        for j0 in range(0, N, tile):
            acc = np.zeros((tile, tile), dtype=np.int64)      # SRAM 위의 psum 타일
            for k0 in range(0, K, tile):
                Xt = X[i0:i0+tile, k0:k0+tile]
                Wt = W[k0:k0+tile, j0:j0+tile]
                Yt, cyc, _, _ = systolic_matmul(Xt, Wt)       # PE 배열 1회 가동
                acc += Yt                                     # SRAM에서 psum 누적
                n_tile_gemm += 1
                total_cycles += cyc
                psum_accum   += tile * tile
            Y[i0:i0+tile, j0:j0+tile] = acc
    return Y, n_tile_gemm, total_cycles, psum_accum

M, K, N, TILE = 8, 8, 8, 4
Xt8 = rng.integers(-4, 5, (M, K))
Wt8 = rng.integers(-4, 5, (K, N))

Y_tiled, n_gemm, cycles, psum_cnt = tiled_systolic_matmul(Xt8, Wt8, TILE)
print("타일링 결과 == numpy 정답 ? →", np.array_equal(Y_tiled, Xt8 @ Wt8))
assert np.array_equal(Y_tiled, Xt8 @ Wt8)

n_formula = (M//TILE) * (N//TILE) * (K//TILE)
print(f"\n타일 GEMM 횟수 : {n_gemm}  (공식 (M/t)(N/t)(K/t) = {n_formula} ✓)")
print(f"총 사이클       : {cycles}   (타일당 {TILE}+{TILE}+{TILE}-2 = {3*TILE-2} 사이클 x {n_gemm}회)")
print(f"psum SRAM 누적  : {psum_cnt}회 — K 방향 타일 수만큼 부분합을 SRAM에서 더함")
print("\n✅ '컴파일러의 Layer→Block 최적화'(Day 2)가 결정하는 것이 바로 이 타일 분할·순서입니다.")

### Step 5-2. 타일 크기가 성능에 미치는 영향 관찰

> ✏️ **직접 해보기:** 아래 셀에서 여러 타일 크기를 비교합니다.
> 타일이 작을수록 파이프라인 충전 오버헤드(타일당 3t-2 사이클 중 정상가동 비중↓)가
> 커지는 것을 확인하세요. 실제 NPU 컴파일러가 타일 크기를 신중히 고르는 이유입니다.

In [ ]:
M = K = N = 16
Xbig = rng.integers(-4, 5, (M, K))
Wbig = rng.integers(-4, 5, (K, N))

print(f"{'타일':>4} {'타일GEMM수':>10} {'총 사이클':>10} {'사이클/MAC 효율':>16}")
for tile in [2, 4, 8, 16]:
    Yt, n_gemm, cycles, _ = tiled_systolic_matmul(Xbig, Wbig, tile)
    assert np.array_equal(Yt, Xbig @ Wbig)
    total_macs = M * K * N
    eff = total_macs / (cycles * tile * tile)   # 가동 사이클 대비 실제 MAC 비율
    print(f"{tile:>4} {n_gemm:>10} {cycles:>10} {eff*100:>14.1f}%")
print("\n💡 타일(=배열)이 클수록 사이클 효율↑ — 단, 실제로는 SRAM 용량이 상한을 정합니다.")

> **✅ Part 5 확인**
> - [ ] 큰 GEMM을 타일로 쪼개고 psum을 SRAM에서 누적하는 과정을 구현했다
> - [ ] 타일 GEMM 횟수 공식 (M/t)(N/t)(K/t)를 확인했다
> - [ ] 타일 크기와 파이프라인 효율의 트레이드오프를 관찰했다

---
# Part 6. INT8 양자화 MAC — 누산기의 비밀과 캘리브레이션

강의 6.2절의 하드웨어 절차를 그대로 코드로 재현합니다.

```text
① 곱셈:    INT8 × INT8 → 최대 16bit
② 누적:    INT32 누산기에 더함  ← 왜 32bit인지 이번에 실험으로 확인!
③ 재양자화: INT32 psum × scale → 다음 레이어용 값 복원
```

### Step 6-1. 양자화 함수 만들기 (scale 결정 = 캘리브레이션의 축소판)

In [ ]:
def quantize(x, num_bits=8, calib_max=None):
    """대칭 양자화. calib_max가 주어지면 그 값을 r_max로 사용(캘리브레이션 흉내).
    반환: (int8 배열, scale)"""
    qmax = 2 ** (num_bits - 1) - 1                     # int8이면 127
    r_max = np.abs(x).max() if calib_max is None else calib_max
    scale = r_max / qmax if r_max > 0 else 1.0
    q = np.clip(np.round(x / scale), -qmax - 1, qmax)   # 범위 밖은 잘림(clipping)!
    return q.astype(np.int8), scale

x_demo = np.array([-1.0, -0.5, 0.0, 0.7, 1.3])
q_demo, s_demo = quantize(x_demo)
print("원본 FP32 :", x_demo)
print("INT8 값   :", q_demo, f"  (scale = {s_demo:.5f})")
print("복원값    :", np.round(q_demo * s_demo, 3), "← 약간의 반올림 오차가 양자화의 대가")

### Step 6-2. INT8 GEMM 전체 파이프라인 — 정확도 손실 측정

FP32 행렬곱을 정답으로 두고, INT8 양자화 → INT32 누산 → 역양자화 경로의 오차를 잽니다.
강의의 "정확도를 약간 양보하고 크기·속도를 크게 얻는다"를 수치로 확인합니다.

In [ ]:
Xf = rng.normal(0, 1.0, (8, 64)).astype(np.float32)    # FP32 입력 (배치 8, 입력 64)
Wf = rng.normal(0, 0.5, (64, 16)).astype(np.float32)   # FP32 가중치 (입력 64, 출력 16)

# ① 양자화 (activation/weight 각자 scale)
qX, sX = quantize(Xf)
qW, sW = quantize(Wf)

# ② INT8 곱 → INT32 누산 (하드웨어와 동일하게 정수로만!) — y = x @ W 형태
acc32 = qX.astype(np.int32) @ qW.astype(np.int32)
print("누산기 dtype:", acc32.dtype, "| 값 범위:", acc32.min(), "~", acc32.max())

# ③ 재양자화(여기선 역양자화로 FP 복원) — scale은 컴파일 시점 고정 상수
Y_int8 = acc32 * (sX * sW)
Y_fp32 = Xf @ Wf

rel_err = np.abs(Y_int8 - Y_fp32).mean() / np.abs(Y_fp32).mean()
print(f"\nFP32 대비 평균 상대 오차: {rel_err*100:.2f}%")
print(f"메모리 절감: FP32 {Xf.nbytes+Wf.nbytes}B → INT8 {qX.nbytes+qW.nbytes}B (4배)")
print("✅ 오차 ~1% 수준으로 메모리 4배·(HW에선)전력 20배를 얻는 거래 — 이것이 양자화입니다.")

### Step 6-3. ★ 누산기 overflow 실험 — "왜 32bit인가"에 대한 결정적 증거

int8 곱 하나는 최대 127×127 = 16,129 (16bit로 충분).
하지만 K번 **누적**하면? K=512라면 최악의 경우 16,129 × 512 ≈ 826만 — int16 최대(32,767)를 한참 넘습니다.
누산기를 일부러 int16으로 줄여서 무슨 일이 나는지 직접 봅시다.

In [ ]:
K = 512
# 최악 조건: 모두 큰 값 (실전에선 드물지만 overflow 원리를 보기 위해)
qX_big = np.full((1, K), 110, dtype=np.int8)   # 활성값 x : (1, K)
qW_big = np.full((K, 1), 120, dtype=np.int8)   # 가중치 W : (K, 1)

correct = (qX_big.astype(np.int64) @ qW_big.astype(np.int64)).item()   # 수학적 정답 (x @ W)

# int16 누산기 (일부러 작게)
acc16 = np.int16(0)
with np.errstate(over="ignore"):
    for k in range(K):
        acc16 = np.int16(acc16 + np.int16(qX_big[0, k]) * np.int16(qW_big[k, 0]))

# int32 누산기 (실제 NPU 방식)
acc32 = np.int32(0)
for k in range(K):
    acc32 = np.int32(acc32 + np.int32(qX_big[0, k]) * np.int32(qW_big[k, 0]))

print(f"수학적 정답      : {correct:,}")
print(f"int16 누산기 결과: {int(acc16):,}   ← 완전히 다른 값 (wrap-around 발생!) 💥")
print(f"int32 누산기 결과: {int(acc32):,}   ← 정답과 일치 ✅")
print()
print(f"int16 최대값 32,767 < 필요값 {correct:,} → overflow가 값을 파괴")
print("💡 강의 6.2절 'K번 누적해도 overflow 안 나도록 32bit 확보'가 이 실험의 결론입니다.")
print("   (Day 5 포팅 실패 #5 '간헐적 NaN/Inf'도 이런 overflow/클리핑 처리 문제에서 옵니다)")

### Step 6-4. 캘리브레이션이 나쁘면 생기는 일 — Day 2·Day 5 실습 연결

캘리브레이션의 본질은 **scale을 정할 r_max를 어떤 데이터로 관찰하느냐**입니다.
실제 데이터 범위는 ±4인데 캘리브레이션 데이터가 ±1 범위만 담고 있었다면?
→ 범위 밖 값이 전부 ±127로 **잘려나가며(clipping)** 정확도가 급락합니다.

이것이 Day 5 포팅 실패 #1 "정확도 급락 — Calibration 데이터 대표성 부족"의 정체입니다.

In [ ]:
# 실제 배포 데이터: outlier 포함, 값 범위 약 ±4
X_real = rng.normal(0, 1.0, (32, 64)).astype(np.float32)   # 입력 (배치 32, 입력 64)
X_real[rng.random(X_real.shape) < 0.03] *= 4.0             # 3% outlier

W_ = rng.normal(0, 0.5, (64, 16)).astype(np.float32)       # 가중치 (입력 64, 출력 16)
qW_, sW_ = quantize(W_)
Y_true = X_real @ W_                                        # y = x @ W

def run_pipeline(calib_max):
    qX_, sX_ = quantize(X_real, calib_max=calib_max)
    clipped = np.mean(np.abs(X_real) > calib_max) * 100
    Y = (qX_.astype(np.int32) @ qW_.astype(np.int32)) * (sX_ * sW_)
    err = np.abs(Y - Y_true).mean() / np.abs(Y_true).mean()
    return err, clipped

good_max = np.abs(X_real).max()        # 좋은 캘리브레이션: 실제 범위를 관찰함
bad_max  = 1.0                         # 나쁜 캘리브레이션: 좁은 범위만 관찰함

err_good, clip_good = run_pipeline(good_max)
err_bad,  clip_bad  = run_pipeline(bad_max)

print(f"{'캘리브레이션':>14} {'r_max':>8} {'잘린 비율':>10} {'상대 오차':>10}")
print(f"{'좋음(대표성O)':>13} {good_max:>8.2f} {clip_good:>9.1f}% {err_good*100:>9.2f}%")
print(f"{'나쁨(대표성X)':>13} {bad_max:>8.2f} {clip_bad:>9.1f}% {err_bad*100:>9.2f}%")
print(f"\n💥 나쁜 캘리브레이션으로 오차 {err_bad/err_good:.0f}배 악화!")
print("✅ '대표성 있는 데이터 300~500장'(Day 2 캘리브레이션 Best Practice)이 왜 중요한지 증명됐습니다.")
print("✏️ 직접 해보기: calib_max를 2.0, 3.0으로 바꿔가며 오차가 어떻게 변하는지 관찰하세요.")

> **✅ Part 6 확인**
> - [ ] INT8 곱 → INT32 누산 → 재양자화의 3단계를 코드로 재현했다
> - [ ] int16 누산기의 overflow를 직접 목격하고 32bit의 필요성을 이해했다
> - [ ] 캘리브레이션 데이터의 대표성이 정확도를 좌우함을 실험으로 확인했다

---
# Part 7. TOPS 계산과 Roofline — 성능 지표 바로 읽기

### Step 7-1. TOPS 공식 구현과 BlackSwan 역산 (강의 7.1절)

```text
TOPS = 2 × (MAC 개수) × (주파수) ÷ 10¹²
```

In [ ]:
def tops(n_mac, freq_hz):
    return 2 * n_mac * freq_hz / 1e12

def n_mac_from_tops(tops_val, freq_hz):
    return tops_val * 1e12 / (2 * freq_hz)

# 강의 연습문제 재현
print("[문제 2] MAC 2,048개 @ 500MHz →", f"{tops(2048, 500e6):.2f} TOPS  (정답 ≈ 2.05)")

# BlackSwan 역산: 2 TOPS @ 300MHz
n = n_mac_from_tops(2.0, 300e6)
print(f"[BlackSwan] 2 TOPS @ 300MHz → MAC ≈ {n:,.0f}개 (Dual Core 합산, XWN 적용 시)")

# 모델 추론 시간 추정: ResNet-50 4.1 GMAC
ops = 2 * 4.1e9
for eff in [1.0, 0.7, 0.4]:
    t_ms = ops / (2.0e12 * eff) * 1000
    print(f"ResNet-50 @ 2 TOPS, 효율 {eff*100:>3.0f}% → {t_ms:>5.1f} ms  ({1000/t_ms:>5.0f} FPS)")
print("\n💡 '이론 TOPS'와 '실측 40~70%'의 차이가 추론 시간에 그대로 반영됩니다 (강의 7.2절).")

### Step 7-2. Roofline 모델 — "FLOPs 절반인데 왜 2배 안 빨라져요?"에 대한 답

실효 성능은 **연산 강도(AI = 연산 수 ÷ 메모리 이동 바이트)**가 결정합니다 (강의 5.2절).

```text
달성 가능 성능 = min( peak_TOPS,  AI × 메모리 대역폭 )
```

가상 NPU(peak 2 TOPS, DRAM 대역폭 4 GB/s 가정) 위에 세 종류의 레이어를 올려 봅니다.

In [ ]:
PEAK_TOPS = 2.0          # TOPS
BW = 4e9                  # bytes/s (DRAM 대역폭 가정)

def layer_profile(name, macs, weight_bytes, act_bytes):
    ops = 2 * macs
    data = weight_bytes + act_bytes           # INT8 가정: 1 원소 = 1 byte
    ai = ops / data                           # ops per byte
    attainable = min(PEAK_TOPS * 1e12, ai * BW)   # ops/s
    return {"name": name, "ai": ai, "perf": attainable / 1e12, "ops": ops}

# 세 레이어 (동일 '느낌'의 규모, 특성만 다르게)
layers = [
    # 3x3 Conv: C=128, 56x56 출력, K=3 → 재사용 많음 = AI 높음
    layer_profile("3x3 Conv",  macs=128*128*9*56*56,
                  weight_bytes=128*128*9, act_bytes=2*128*56*56),
    # FC 4096→4096: weight가 크고 재사용 없음 = AI 낮음
    layer_profile("FC 4096",   macs=4096*4096,
                  weight_bytes=4096*4096, act_bytes=2*4096),
    # Depthwise 3x3: 채널당 연산 → 연산 적고 데이터 이동 비중 큼
    layer_profile("DW-Conv",   macs=256*9*56*56,
                  weight_bytes=256*9, act_bytes=2*256*56*56),
]

# Roofline 그리기
ai_axis = np.logspace(-1, 4, 200)
roof = np.minimum(PEAK_TOPS, ai_axis * BW / 1e12)

plt.figure(figsize=(8.5, 5))
plt.loglog(ai_axis, roof, "k-", lw=2, label="Roofline: min(peak, AI×BW)")
plt.axhline(PEAK_TOPS, ls=":", c="gray")
ridge = PEAK_TOPS * 1e12 / BW
plt.axvline(ridge, ls="--", c="gray", alpha=0.6)
plt.text(ridge*1.1, 0.02, f"ridge point\nAI={ridge:.0f}", fontsize=9)

for L in layers:
    plt.plot(L["ai"], L["perf"], "o", ms=11)
    plt.annotate(f"{L['name']}\nAI={L['ai']:.0f} → {L['perf']:.2f} TOPS",
                 (L["ai"], L["perf"]), textcoords="offset points", xytext=(8, -14), fontsize=9)

plt.xlabel("연산 강도 AI (ops / byte)"); plt.ylabel("달성 가능 성능 (TOPS)")
plt.title("가상 NPU Roofline — 2 TOPS peak, 4 GB/s DRAM")
plt.grid(True, which="both", alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

print(f"{'레이어':<10}{'AI':>8}{'달성 TOPS':>11}{'peak 대비':>10}   병목")
for L in layers:
    bound = "Compute-bound ✅" if L["ai"] >= ridge else "Memory-bound ⚠️ (MAC이 놈)"
    print(f"{L['name']:<10}{L['ai']:>8.0f}{L['perf']:>11.2f}{L['perf']/PEAK_TOPS*100:>9.0f}%   {bound}")
print("\n✅ FC·DW-Conv는 연산을 줄여도 빨라지지 않는다 — 병목이 메모리이기 때문 (강의 5.2절).")

> **✅ Part 7 확인**
> - [ ] TOPS 공식으로 MAC 개수·추론 시간을 계산할 수 있다
> - [ ] Roofline에서 ridge point 좌/우가 Memory/Compute-bound임을 설명할 수 있다
> - [ ] "FLOPs 절반 ≠ 속도 2배"의 이유를 그래프로 지목할 수 있다
>
> ✏️ **직접 해보기:** `BW`를 8e9(대역폭 2배)로 올리면 어떤 레이어가 이득을 보나요? peak를 4 TOPS로 올리면?

---
# Part 8. Fallback 비용 시뮬레이션 — NPU 활용률이 성능을 지배한다

강의 8.2절: Fallback이 느린 진짜 이유는 CPU가 느려서가 아니라
**NPU↔CPU 왕복 데이터 이동이 파이프라인을 끊기 때문**입니다.
GELU 하나가 CPU로 떨어진 모델과, ReLU로 교체한 모델의 지연을 비교합니다.

### Step 8-1. 레이어 파이프라인 지연 모델

In [ ]:
TRANSFER_MS = 1.5    # NPU↔CPU 방향 전환 1회당 데이터 이동 비용 (ms)

def pipeline_latency(layers):
    """layers: [(이름, 장치 'NPU'|'CPU', 실행시간ms), ...]
    장치가 바뀔 때마다 전송 비용 추가. 반환: 총지연, NPU활용률, 전송횟수, 타임라인"""
    total, transfers, npu_time = 0.0, 0, 0.0
    timeline, prev = [], "NPU"          # 입력은 NPU 쪽에 있다고 가정
    for name, dev, t in layers:
        if dev != prev:
            total += TRANSFER_MS; transfers += 1
            timeline.append((f"↔ 전송", TRANSFER_MS, "XFER"))
        total += t
        if dev == "NPU": npu_time += t
        timeline.append((name, t, dev))
        prev = dev
    return total, npu_time / total * 100, transfers, timeline

# 모델 A: GELU가 NPU 미지원 → CPU Fallback (강의 8.3절 시나리오)
model_fallback = [
    ("Conv1", "NPU", 2.0), ("Conv2", "NPU", 3.0),
    ("GELU",  "CPU", 4.0),                      # ← Fallback!
    ("Conv3", "NPU", 3.0), ("Conv4", "NPU", 2.0),
    ("GELU2", "CPU", 4.0),                      # ← 또 Fallback!
    ("Conv5", "NPU", 2.5), ("FC",    "NPU", 1.0),
]
# 모델 B: GELU → ReLU 교체 (Fallback 4대 대응 ① '연산자 교체')
model_fixed = [(n.replace("GELU2","ReLU2").replace("GELU","ReLU"),
                "NPU" if d == "CPU" else d,
                0.3 if "GELU" in n else t) for n, d, t in model_fallback]

for title, m in [("A. GELU Fallback 모델", model_fallback), ("B. ReLU 교체 모델", model_fixed)]:
    total, util, xfers, _ = pipeline_latency(m)
    print(f"{title:<22} 총 {total:5.1f} ms | NPU 활용률 {util:5.1f}% | 장치 전환 {xfers}회 | {1000/total:.0f} FPS")

tA, _, _, tlA = pipeline_latency(model_fallback)
tB, _, _, _   = pipeline_latency(model_fixed)
print(f"\n💡 연산자 교체 하나로 지연 {tA:.1f} → {tB:.1f} ms ({tA/tB:.1f}배 개선)")
print("   순수 GELU 실행시간(8ms)보다 전송비용(6ms)이 문제의 절반 — '왕복이 적'이라는 증거입니다.")

### Step 8-2. 타임라인 시각화 — 전송 구간이 파이프라인을 끊는 모습

In [ ]:
def draw_timeline(layers, title, ax):
    _, _, _, tl = pipeline_latency(layers)
    colors = {"NPU": "#4c72b0", "CPU": "#c44e52", "XFER": "#999999"}
    t = 0
    for name, dur, dev in tl:
        ax.barh(0, dur, left=t, color=colors[dev], edgecolor="white")
        if dur > 0.8:
            ax.text(t + dur/2, 0, name, ha="center", va="center", color="white", fontsize=8)
        t += dur
    ax.set_yticks([]); ax.set_xlim(0, 25); ax.set_xlabel("시간 (ms)")
    ax.set_title(f"{title}  — 총 {t:.1f} ms", loc="left", fontsize=10)

fig, axes = plt.subplots(2, 1, figsize=(10, 3.2))
draw_timeline(model_fallback, "A. GELU가 CPU로 Fallback (회색=NPU↔CPU 전송)", axes[0])
draw_timeline(model_fixed,    "B. GELU→ReLU 교체 후 (전 구간 NPU)", axes[1])
plt.tight_layout(); plt.show()

print("✅ Day 2 체크포인트 ③ 'NPU 활용률 90% 이상'이 왜 목표인지 눈으로 확인했습니다.")
print("✏️ 직접 해보기: 두 GELU를 모델 '맨 끝'으로 몰면(전환 횟수↓) 총지연이 얼마나 줄까요?")
print("   → Fallback 4대 대응 ③ '구간을 몰아 왕복 최소화' 전략을 직접 검증해 보세요.")

> **✅ Part 8 확인**
> - [ ] Fallback 비용 = CPU 실행시간 + **왕복 전송비용**임을 분해해서 설명할 수 있다
> - [ ] 연산자 교체(GELU→ReLU)의 효과를 정량적으로 확인했다
> - [ ] NPU 활용률과 총지연의 관계를 이해했다

---
# Part 9. 🏁 종합 미니 프로젝트 — 나만의 가상 NPU로 Conv 레이어 End-to-End 실행

지금까지 만든 부품을 전부 조립합니다. **이 흐름은 Day 2 컴파일 파이프라인 그 자체**입니다.

```text
FP32 Conv 레이어
  → [양자화]      Part 6의 quantize            (컴파일러의 캘리브레이션·상수 주입)
  → [im2col]     Part 2의 im2col               (컴파일러의 그래프 변환 — 패치가 행)
  → [systolic]   Part 3의 systolic_matmul      (NPU MAC 배열에서 Y = X_col @ W_mat 실행)
  → [재양자화]    scale 곱                      (하드웨어 requantize)
  → [성능 추정]   사이클 → 시간 → FPS           (Day 1 check_latency.sh가 재는 것)
```

### Step 9-1. 가상 NPU 클래스로 조립하기

In [ ]:
class VirtualNPU:
    """교육용 가상 NPU: INT8 systolic array + 사이클/에너지 추정"""
    def __init__(self, freq_mhz=300, name="MyNPU"):
        self.freq = freq_mhz * 1e6
        self.name = name

    def run_conv(self, X_fp, W_fp, calib_max=None):
        OC, C, KH, KW = W_fp.shape
        # ① 양자화 (캘리브레이션 → scale 고정)
        qW, sW = quantize(W_fp.reshape(OC, -1))
        qX_img, sX = quantize(X_fp, calib_max=calib_max)
        # ② im2col (컴파일러의 그래프 변환) — 패치가 행: (OH·OW, C·K·K)
        X_col, OH, OW = im2col(qX_img.astype(np.int64), KH, KW)
        W_mat = qW.astype(np.int64).T              # (C·K·K, OC) — y = x @ W 형태
        # ③ systolic array 실행 (INT 연산, 가중치 W_mat이 PE에 고정)
        acc, cycles, mpc, _ = systolic_matmul(X_col, W_mat)
        # ④ 재양자화 (INT32 psum → FP 복원; 실제 HW는 다음 레이어 INT8로)
        Y = (acc * (sW * sX)).T.reshape(OC, OH, OW)
        # ⑤ 성능 리포트
        t_ms = cycles / self.freq * 1000
        total_macs = sum(mpc)
        util = total_macs / (cycles * W_mat.shape[0] * W_mat.shape[1])
        return Y, {"cycles": cycles, "time_ms": t_ms, "fps_est": 1000 / t_ms,
                   "macs": total_macs, "pe_array": W_mat.shape, "utilization": util}

# ── 레이어 정의: 입력 (3, 16, 16), 커널 (8, 3, 3, 3) ──
X_fp = rng.normal(0, 1, (3, 16, 16)).astype(np.float32)
W_fp = rng.normal(0, 0.3, (8, 3, 3, 3)).astype(np.float32)

npu = VirtualNPU(freq_mhz=300, name="BlackSwan-Sim")
Y_npu, report = npu.run_conv(X_fp, W_fp)
Y_gold = conv2d_direct(X_fp.astype(np.float64), W_fp.astype(np.float64))

err = np.abs(Y_npu - Y_gold).mean() / np.abs(Y_gold).mean()
print(f"═══ {npu.name} 실행 리포트 ═══")
print(f"PE 배열       : {report['pe_array'][0]} x {report['pe_array'][1]} = {report['pe_array'][0]*report['pe_array'][1]} MAC (가중치 고정)")
print(f"총 사이클     : {report['cycles']:,}")
print(f"총 MAC        : {report['macs']:,}")
print(f"활용률        : {report['utilization']*100:.1f}%")
print(f"레이어 지연   : {report['time_ms']*1000:.1f} µs @ 300MHz")
print(f"환산 FPS      : {report['fps_est']:,.0f} (이 레이어만 반복 시)")
print(f"FP32 대비 오차: {err*100:.2f}%  (양자화 비용)")
print("\n✅ 축하합니다! 양자화→im2col→systolic→재양자화 전 과정을 가진 NPU를 완성했습니다.")

### Step 9-2. 최종 과제 — 성능 비교 리포트 (Day 1 리포트 양식의 가상 버전)

아래 표를 직접 채우세요. 코드는 힌트만 제공합니다.

| 실험 | 조건 | 사이클 | 지연(µs) | 오차(%) |
| --- | --- | --- | --- | --- |
| 기준 | 위 설정 그대로 | | | |
| 실험1 | 입력을 (3, 32, 32)로 | | | |
| 실험2 | 출력 채널 OC=16으로 | | | |
| 실험3 | `calib_max=0.5` (나쁜 캘리브레이션) | | | |
| 실험4 | 주파수 600MHz | | | |

**분석 질문 (2~3문장씩):**
1. 실험1에서 사이클이 몇 배 늘었나요? 입력 면적 4배와 어떤 관계인가요? (힌트: N = OH·OW)
2. 실험3에서 지연은 그대로인데 오차만 커진 이유는?
3. 실험4에서 지연이 정확히 절반이 되나요? 실제 NPU에서는 왜 안 될 수도 있나요? (힌트: 메모리 대역폭은 그대로)

In [ ]:
# ✏️ 여기에 실험 코드를 작성하세요. 예시 (실험1):
X_big = rng.normal(0, 1, (3, 32, 32)).astype(np.float32)
Y1, r1 = npu.run_conv(X_big, W_fp)
print(f"실험1: 사이클 {r1['cycles']:,} | 지연 {r1['time_ms']*1000:.1f} µs")
print(f"기준 대비 사이클 비율: {r1['cycles']/report['cycles']:.2f}배")

# 실험2~4는 직접 작성해 보세요!
# 힌트: 실험2 → W_fp16 = rng.normal(0, 0.3, (16, 3, 3, 3)).astype(np.float32)
# 힌트: 실험3 → npu.run_conv(X_fp, W_fp, calib_max=0.5) 후 오차 재계산
# 힌트: 실험4 → npu_fast = VirtualNPU(freq_mhz=600)

---
# Part 10. 정리 — 오늘 만든 것과 실물 NPU의 관계

## 오늘 구현한 것 ↔ 실물 시스템 대응표

| 오늘의 코드 | 실물 대응 (본 부트캠프) |
| --- | --- |
| `mac()` 함수 | BlackSwan NPU의 MAC 유닛 ~3,300개 중 하나 |
| `im2col()` | 디퍼아이 컴파일러의 그래프 변환 단계 |
| `systolic_matmul()` | NPU MAC 배열의 사이클 단위 동작 |
| `tiled_systolic_matmul()` | 컴파일러의 Layer→Block 타일링 (`.tachyrt` 안에 스케줄로 박제) |
| `quantize()`의 calib_max | Day 2 캘리브레이션 데이터 300~500장이 결정하는 scale |
| int32 누산기 실험 | "간헐적 NaN/Inf" 포팅 실패의 근본 원인 |
| Roofline 그래프 | "FLOPs 줄였는데 FPS 안 나와요"를 진단하는 도구 |
| `pipeline_latency()` | NPU Profiler가 재는 Fallback 비용, 활용률 90% 목표의 근거 |
| `VirtualNPU.run_conv()` | Day 1 `check_latency.sh` / `check_throughput.sh`가 측정하는 것 |

## 오늘의 시뮬레이터가 생략한 것 (실물은 여기가 더 어렵다)

- DMA 더블 버퍼링 (전송과 연산의 겹침), SRAM 뱅크 충돌
- Conv+BN+ReLU **fusion** — 오늘은 레이어를 따로 돌렸지만 실물은 융합해 메모리 왕복 제거
- Per-channel scale (오늘은 per-tensor만), zero-point(비대칭 양자화)
- 프루닝 스킵 로직 (BlackSwan Pruning Engine이 하드웨어로 처리)

## ✏️ 심화 도전 과제 (선택)

1. **Output-Stationary 시뮬레이터**를 구현하고 Part 4의 접근 횟수를 WS와 비교하기
2. `systolic_matmul`에 **프루닝 스킵**을 추가하기: `W[k,j]==0`이면 MAC을 세지 않도록 수정하고, 50% 구조적 프루닝 시 유효 MAC이 어떻게 줄어드는지 측정 (BlackSwan Pruning Engine의 원리)
3. Part 8의 모델에서 **Fallback 구간 재배치 최적화**를 자동으로 찾는 함수 작성하기
4. `VirtualNPU`에 Part 4의 에너지 모델을 통합해 **레이어당 에너지(상대값)**까지 리포트에 추가하기

---

수고하셨습니다! 🎉 이제 여러분은 NPU를 "블랙박스"가 아니라
**"사이클 단위로 머릿속에서 돌려볼 수 있는 기계"**로 이해하게 되었습니다.
이 감각은 Day 2의 컴파일 로그를 읽을 때, Day 3의 병목을 찾을 때, Day 5의 벤치마크를 해석할 때 그대로 쓰입니다.
